<a href="https://colab.research.google.com/github/Nausheen1295/NorthStar-Coursework/blob/main/notebooks/Section_1_R/01_sql_in_r.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [32]:
install.packages("DBI")
install.packages("RSQLite")
install.packages("dplyr")
install.packages("sqldf")

library(DBI)
library(RSQLite)
library(dplyr)
library(sqldf)

Installing package into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)

Installing package into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)

Installing package into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)

Installing package into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)



In [33]:
orders <- read.csv("/content/orders.csv")
deliveries <- read.csv("/content/deliveries.csv")
customers <- read.csv("/content/customers.csv")
drivers <- read.csv("/content/drivers.csv")
vehicles <- read.csv("/content/vehicles.csv")
hubs <- read.csv("/content/hubs.csv")
complaints <- read.csv("/content/complaints.csv")
incidents <- read.csv("/content/incidents.csv")
app_events <- read.csv("/content/app_events.csv")

head(orders)

,order_id,customer_id,service_type,order_created_at,promised_window_hours,pickup_zone,dropoff_zone,priority_level,order_value,booking_channel,special_handling_flag
,<chr>,<chr>,<chr>,<chr>,<int>,<chr>,<chr>,<chr>,<dbl>,<chr>,<int>
1,O00001,C0292,Passenger,2024-08-20 14:43:00,6,Airport,South,Medium,126.65,App,0
2,O00002,C0459,Passenger,2024-05-14 22:16:00,24,North,AIRPORT,Low,109.30,App,0
3,O00003,C0161,Passenger,2025-09-02 14:37:00,4,West,AIRPORT,High,33.50,Phone,0
4,O00004,C0520,Parcel,2025-01-11 17:15:00,2,RiverSide,North,Medium,10.04,App,1
5,O00005,C0558,Retail,2025-02-17 19:32:00,12,Riverside,SOUTH,Low,125.58,Phone,0
6,O00006,C0437,Retail,2024-08-05 04:55:00,1,CENTRAL,East,High,151.44,Web,1


In [34]:
# Check the structure of each table
str(orders)
str(deliveries)
str(customers)
str(drivers)
str(vehicles)
str(hubs)
str(complaints)
str(incidents)
str(app_events)

'data.frame':	1250 obs. of  11 variables:
 $ order_id             : chr  "O00001" "O00002" "O00003" "O00004" ...
 $ customer_id          : chr  "C0292" "C0459" "C0161" "C0520" ...
 $ service_type         : chr  "Passenger" "Passenger" "Passenger" "Parcel" ...
 $ order_created_at     : chr  "2024-08-20 14:43:00" "2024-05-14 22:16:00" "2025-09-02 14:37:00" "2025-01-11 17:15:00" ...
 $ promised_window_hours: int  6 24 4 2 12 1 2 4 12 6 ...
 $ pickup_zone          : chr  "Airport" "North" "West" "RiverSide" ...
 $ dropoff_zone         : chr  "South" "AIRPORT" "AIRPORT" "North" ...
 $ priority_level       : chr  "Medium" "Low" "High" "Medium" ...
 $ order_value          : num  126.7 109.3 33.5 10 125.6 ...
 $ booking_channel      : chr  "App" "App" "Phone" "App" ...
 $ special_handling_flag: int  0 0 0 1 0 1 0 0 0 0 ...
'data.frame':	950 obs. of  13 variables:
 $ delivery_id                  : chr  "DL00001" "DL00002" "DL00003" "DL00004" ...
 $ order_id                     : chr  "O00938" "

In [35]:
# Check missing values in each table
colSums(is.na(orders))
colSums(is.na(deliveries))
colSums(is.na(customers))
colSums(is.na(drivers))
colSums(is.na(vehicles))
colSums(is.na(hubs))
colSums(is.na(complaints))
colSums(is.na(incidents))
colSums(is.na(app_events))

order_id           customer_id          service_type 
                    0                     0                     0 
     order_created_at promised_window_hours           pickup_zone 
                    0                     0                     0 
         dropoff_zone        priority_level           order_value 
                    0                     0                     0 
      booking_channel special_handling_flag 
                    0                     0

delivery_id                      order_id 
                            0                             0 
                    driver_id                    vehicle_id 
                            0                             0 
                       hub_id                 dispatch_time 
                            0                             0 
        delivery_completed_at               delivery_status 
                            0                             0 
            route_distance_km   manual_route_override_count 
                            0                             0 
  proof_of_completion_missing customer_rating_post_delivery 
                            0                            14 
          fuel_or_charge_cost 
                            0

customer_id                  age            home_zone 
                   0                    0                    0 
       customer_type          signup_date        loyalty_score 
                   0                    0                   20 
app_engagement_score    preferred_channel       account_status 
                   0                    0                    0

driver_id        base_zone  employment_type years_experience 
               0                0                0                0 
  training_score    driver_rating shift_preference      active_flag 
               7                0                0                0

vehicle_id       vehicle_type      assigned_zone    commission_date 
                 0                  0                  0                  0 
battery_health_pct        odometer_km maintenance_status telematics_version 
                 4                  0                  0                  0

hub_id       hub_name           zone       hub_type capacity_score 
             0              0              0              0              0

complaint_id         customer_id            order_id      complaint_type 
                  0                   0                   0                   0 
            channel            severity          created_at              status 
                  0                   0                   0                   0 
    resolution_days compensation_amount 
                  0                  16

incident_id       delivery_id     incident_type       reported_at 
                0                 0                 0                 0 
         severity resolution_status    resolved_hours 
                0                 0                17

event_id     customer_id        order_id event_timestamp      event_type 
              0               0               0               0               0 
     session_id     device_type    zone_context  api_latency_ms    success_flag 
              0               0               0               0               0

In [36]:
# Create an in-memory SQLite database
conn <- dbConnect(SQLite(), ":memory:")

# Write CSV dataframes as SQL tables
dbWriteTable(conn, "orders", orders, overwrite = TRUE)
dbWriteTable(conn, "deliveries", deliveries, overwrite = TRUE)
dbWriteTable(conn, "customers", customers, overwrite = TRUE)
dbWriteTable(conn, "drivers", drivers, overwrite = TRUE)
dbWriteTable(conn, "vehicles", vehicles, overwrite = TRUE)
dbWriteTable(conn, "hubs", hubs, overwrite = TRUE)
dbWriteTable(conn, "complaints", complaints, overwrite = TRUE)
dbWriteTable(conn, "incidents", incidents, overwrite = TRUE)
dbWriteTable(conn, "app_events", app_events, overwrite = TRUE)

# List all tables
dbListTables(conn)

[1] "app_events" "complaints" "customers"  "deliveries" "drivers"   
[6] "hubs"       "incidents"  "orders"     "vehicles"

In [37]:
# Select first 10 orders
select_query <- "
SELECT order_id, customer_id, service_type, pickup_zone, dropoff_zone, order_value
FROM orders
LIMIT 10;
"

select_result <- dbGetQuery(conn, select_query)
select_result

order_id,customer_id,service_type,pickup_zone,dropoff_zone,order_value
<chr>,<chr>,<chr>,<chr>,<chr>,<dbl>
O00001,C0292,Passenger,Airport,South,126.65
O00002,C0459,Passenger,North,AIRPORT,109.30
O00003,C0161,Passenger,West,AIRPORT,33.50
O00004,C0520,Parcel,RiverSide,North,10.04
O00005,C0558,Retail,Riverside,SOUTH,125.58
O00006,C0437,Retail,CENTRAL,East,151.44
O00007,C0001,Business,CENTRAL,Airport,76.12
O00008,C0157,Parcel,Riverside,Riverside,35.06
O00009,C0141,Retail,NORTH,East,78.93


In [38]:
# Insert a new record into orders table
insert_query <- "
INSERT INTO orders
(order_id, customer_id, service_type, order_created_at, promised_window_hours, pickup_zone, dropoff_zone, priority_level, order_value, booking_channel, special_handling_flag)
VALUES
('O99999', 'C9999', 'LastMile', '2026-05-09 10:00:00', 4, 'Central', 'North', 'High', 85.50, 'App', 1);
"

dbExecute(conn, insert_query)

[1] 1

In [39]:
# Check inserted record
dbGetQuery(conn, "
SELECT *
FROM orders
WHERE order_id = 'O99999';
")

order_id,customer_id,service_type,order_created_at,promised_window_hours,pickup_zone,dropoff_zone,priority_level,order_value,booking_channel,special_handling_flag
<chr>,<chr>,<chr>,<chr>,<int>,<chr>,<chr>,<chr>,<dbl>,<chr>,<int>
O99999,C9999,LastMile,2026-05-09 10:00:00,4,Central,North,High,85.5,App,1


In [40]:
# Update the order value of the inserted order
update_query <- "
UPDATE orders
SET order_value = 95.00,
    priority_level = 'Urgent'
WHERE order_id = 'O99999';
"

dbExecute(conn, update_query)

[1] 1

In [41]:
dbGetQuery(conn, "
SELECT order_id, order_value, priority_level
FROM orders
WHERE order_id = 'O99999';
")

order_id,order_value,priority_level
<chr>,<dbl>,<chr>
O99999,95,Urgent


In [42]:
# Delete the inserted test order
delete_query <- "
DELETE FROM orders
WHERE order_id = 'O99999';
"

dbExecute(conn, delete_query)

[1] 1

In [43]:
dbGetQuery(conn, "
SELECT *
FROM orders
WHERE order_id = 'O99999';
")

order_id,customer_id,service_type,order_created_at,promised_window_hours,pickup_zone,dropoff_zone,priority_level,order_value,booking_channel,special_handling_flag
<chr>,<chr>,<chr>,<chr>,<int>,<chr>,<chr>,<chr>,<dbl>,<chr>,<int>


In [44]:
#Total revenue by service type
revenue_query <- "
SELECT
    service_type,
    COUNT(order_id) AS total_orders,
    ROUND(SUM(order_value), 2) AS total_revenue,
    ROUND(AVG(order_value), 2) AS average_order_value,
    ROUND(MIN(order_value), 2) AS minimum_order_value,
    ROUND(MAX(order_value), 2) AS maximum_order_value
FROM orders
GROUP BY service_type
ORDER BY total_revenue DESC;
"

revenue_result <- dbGetQuery(conn, revenue_query)
revenue_result

service_type,total_orders,total_revenue,average_order_value,minimum_order_value,maximum_order_value
<chr>,<int>,<dbl>,<dbl>,<dbl>,<dbl>
Passenger,341,32761.11,96.07,5.92,326.38
Parcel,308,26985.62,87.62,3.57,510.06
Retail,297,26734.06,90.01,4.22,355.62
Business,165,15220.43,92.25,6.28,321.68
Medical,139,12111.93,87.14,2.04,292.33


In [45]:
#Estimated profit by service type
profit_query <- "
SELECT
    o.service_type,
    COUNT(o.order_id) AS total_orders,
    ROUND(SUM(o.order_value), 2) AS total_revenue,
    ROUND(SUM(d.fuel_or_charge_cost), 2) AS total_delivery_cost,
    ROUND(SUM(o.order_value - d.fuel_or_charge_cost), 2) AS estimated_profit
FROM orders o
JOIN deliveries d
ON o.order_id = d.order_id
GROUP BY o.service_type
ORDER BY estimated_profit ASC;
"

profit_result <- dbGetQuery(conn, profit_query)
profit_result

service_type,total_orders,total_revenue,total_delivery_cost,estimated_profit
<chr>,<int>,<dbl>,<dbl>,<dbl>
Medical,108,9344.88,1379.48,7965.40
Business,126,12279.23,1655.91,10623.32
Retail,224,19444.86,2906.27,16538.59
Parcel,230,20735.44,3009.01,17726.43
Passenger,262,25463.36,3248.56,22214.80


In [46]:
#Average manual route override by hub
override_query <- "
SELECT
    h.hub_name,
    h.zone,
    COUNT(d.delivery_id) AS total_deliveries,
    ROUND(AVG(d.manual_route_override_count), 2) AS average_route_overrides
FROM deliveries d
JOIN hubs h
ON d.hub_id = h.hub_id
GROUP BY h.hub_name, h.zone
ORDER BY average_route_overrides DESC;
"

override_result <- dbGetQuery(conn, override_query)
override_result

hub_name,zone,total_deliveries,average_route_overrides
<chr>,<chr>,<int>,<dbl>
Midtown Relay,Central,128,1.11
Riverside Hub,Riverside,115,1.05
North Exchange,North,136,1.03
Central Core,Central,115,0.95
South Link,South,106,0.92
Airport Hub,Airport,104,0.91
East Dock,East,119,0.89
West Gate,West,127,0.87


In [47]:
#Which hubs have the highest failed deliveries?
hub_failure_query <- "
SELECT
    h.hub_id,
    h.hub_name,
    h.zone,
    COUNT(d.delivery_id) AS total_deliveries,
    SUM(CASE WHEN d.delivery_status = 'Failed' THEN 1 ELSE 0 END) AS failed_deliveries,
    ROUND(
        SUM(CASE WHEN d.delivery_status = 'Failed' THEN 1 ELSE 0 END) * 100.0 / COUNT(d.delivery_id),
        2
    ) AS failure_rate_percentage
FROM deliveries d
JOIN hubs h
ON d.hub_id = h.hub_id
GROUP BY h.hub_id, h.hub_name, h.zone
ORDER BY failure_rate_percentage DESC;
"

hub_failure_result <- dbGetQuery(conn, hub_failure_query)
hub_failure_result

hub_id,hub_name,zone,total_deliveries,failed_deliveries,failure_rate_percentage
<chr>,<chr>,<chr>,<int>,<int>,<dbl>
H08,Midtown Relay,Central,128,26,20.31
H05,Central Core,Central,115,23,20.00
H06,Airport Hub,Airport,104,15,14.42
H04,West Gate,West,127,16,12.60
H01,North Exchange,North,136,17,12.50
H07,Riverside Hub,Riverside,115,14,12.17
H02,South Link,South,106,10,9.43
H03,East Dock,East,119,11,9.24


In [48]:
#Which service type has the most complaints?
complaint_service_query <- "
SELECT
    o.service_type,
    COUNT(c.complaint_id) AS total_complaints,
    ROUND(AVG(c.compensation_amount), 2) AS average_compensation,
    ROUND(SUM(c.compensation_amount), 2) AS total_compensation
FROM orders o
LEFT JOIN complaints c
ON o.order_id = c.order_id
GROUP BY o.service_type
ORDER BY total_complaints DESC;
"

complaint_service_result <- dbGetQuery(conn, complaint_service_query)
complaint_service_result

service_type,total_complaints,average_compensation,total_compensation
<chr>,<int>,<dbl>,<dbl>
Passenger,84,20.72,1657.53
Retail,83,19.05,1486.14
Parcel,77,20.61,1463.65
Business,39,20.79,810.86
Medical,37,20.56,740.01


In [49]:
#Do route overrides affect customer ratings?
route_rating_query <- "
SELECT
    d.manual_route_override_count,
    COUNT(d.delivery_id) AS total_deliveries,
    ROUND(AVG(d.customer_rating_post_delivery), 2) AS average_customer_rating,
    SUM(CASE WHEN d.delivery_status = 'Failed' THEN 1 ELSE 0 END) AS failed_deliveries
FROM deliveries d
GROUP BY d.manual_route_override_count
ORDER BY d.manual_route_override_count DESC;
"

route_rating_result <- dbGetQuery(conn, route_rating_query)
route_rating_result

manual_route_override_count,total_deliveries,average_customer_rating,failed_deliveries
<int>,<int>,<dbl>,<int>
7,1,4.01,0
5,7,3.38,0
4,23,3.87,3
3,57,3.67,10
2,153,3.82,22
1,310,3.88,51
0,399,3.90,46


In [50]:
#Which drivers have the highest failed delivery rate?
driver_failure_query <- "
SELECT
    dr.driver_id,
    dr.base_zone,
    dr.years_experience,
    dr.training_score,
    dr.driver_rating,
    COUNT(d.delivery_id) AS total_deliveries,
    SUM(CASE WHEN d.delivery_status = 'Failed' THEN 1 ELSE 0 END) AS failed_deliveries,
    ROUND(
        SUM(CASE WHEN d.delivery_status = 'Failed' THEN 1 ELSE 0 END) * 100.0 / COUNT(d.delivery_id),
        2
    ) AS failure_rate_percentage
FROM drivers dr
JOIN deliveries d
ON dr.driver_id = d.driver_id
GROUP BY dr.driver_id, dr.base_zone, dr.years_experience, dr.training_score, dr.driver_rating
HAVING COUNT(d.delivery_id) >= 3
ORDER BY failure_rate_percentage DESC
LIMIT 10;
"

driver_failure_result <- dbGetQuery(conn, driver_failure_query)
driver_failure_result

driver_id,base_zone,years_experience,training_score,driver_rating,total_deliveries,failed_deliveries,failure_rate_percentage
<chr>,<chr>,<int>,<dbl>,<dbl>,<int>,<int>,<dbl>
D063,north,12,85.7,4.03,3,2,66.67
D092,East,15,88.2,4.24,5,3,60.00
D104,WEST,15,87.7,3.45,7,4,57.14
D024,RiverSide,8,71.4,3.35,8,4,50.00
D103,Central,15,72.5,4.40,4,2,50.00
D111,Airport,15,79.2,4.12,4,2,50.00
D132,South,8,77.6,4.20,4,2,50.00
D170,West,14,75.2,4.48,4,2,50.00
D010,West,8,70.0,3.95,7,3,42.86


In [51]:
#Which vehicles are linked to more incidents?
vehicle_incident_query <- "
SELECT
    v.vehicle_id,
    v.vehicle_type,
    v.assigned_zone,
    v.battery_health_pct,
    v.odometer_km,
    v.maintenance_status,
    COUNT(i.incident_id) AS total_incidents
FROM vehicles v
LEFT JOIN deliveries d
ON v.vehicle_id = d.vehicle_id
LEFT JOIN incidents i
ON d.delivery_id = i.delivery_id
GROUP BY v.vehicle_id, v.vehicle_type, v.assigned_zone, v.battery_health_pct, v.odometer_km, v.maintenance_status
ORDER BY total_incidents DESC
LIMIT 10;
"

vehicle_incident_result <- dbGetQuery(conn, vehicle_incident_query)
vehicle_incident_result

vehicle_id,vehicle_type,assigned_zone,battery_health_pct,odometer_km,maintenance_status,total_incidents
<chr>,<chr>,<chr>,<dbl>,<int>,<chr>,<int>
V047,EV,Ctr,93.7,134347,Scheduled,9
V108,Diesel,AIRPORT,54.6,141290,InRepair,7
V030,CargoVan,North,78.0,134360,Active,6
V046,EV,north,95.8,101425,Active,6
V097,EV,Ctr,92.1,18680,Active,6
V005,CargoVan,West,58.6,146638,Active,5
V009,CargoVan,South,68.8,156687,Active,5
V035,CargoVan,East,83.6,73804,Active,5
V042,EV,East,80.5,215870,InRepair,5


In [52]:
#Which zones create the highest compensation cost?
zone_compensation_query <- "
SELECT
    o.pickup_zone,
    COUNT(c.complaint_id) AS total_complaints,
    ROUND(SUM(c.compensation_amount), 2) AS total_compensation,
    ROUND(AVG(c.resolution_days), 2) AS average_resolution_days
FROM orders o
JOIN complaints c
ON o.order_id = c.order_id
GROUP BY o.pickup_zone
ORDER BY total_compensation DESC;
"

zone_compensation_result <- dbGetQuery(conn, zone_compensation_query)
zone_compensation_result

pickup_zone,total_complaints,total_compensation,average_resolution_days
<chr>,<int>,<dbl>,<dbl>
West,25,541.94,7.96
CENTRAL,26,540.53,8.85
EAST,24,526.95,6.04
East,26,485.35,7.54
South,25,461.15,8.68
Riverside,24,444.74,8.83
Central,16,425.16,6.63
north,20,389.17,8.30
RiverSide,21,380.34,6.48


In [53]:
#Which customer type creates the most complaints?
customer_complaint_query <- "
SELECT
    cu.customer_type,
    COUNT(c.complaint_id) AS total_complaints,
    ROUND(AVG(c.compensation_amount), 2) AS average_compensation,
    ROUND(SUM(c.compensation_amount), 2) AS total_compensation
FROM customers cu
JOIN complaints c
ON cu.customer_id = c.customer_id
GROUP BY cu.customer_type
ORDER BY total_complaints DESC;
"

customer_complaint_result <- dbGetQuery(conn, customer_complaint_query)
customer_complaint_result

customer_type,total_complaints,average_compensation,total_compensation
<chr>,<int>,<dbl>,<dbl>
Consumer,242,20.62,4764.11
SME,50,18.47,868.15
Enterprise,28,20.23,525.93


In [54]:
# Which complaint types have the highest compensation cost?

complaint_cost_query <- "
SELECT
    complaint_type,
    COUNT(complaint_id) AS total_complaints,
    ROUND(SUM(compensation_amount), 2) AS total_compensation,
    ROUND(AVG(compensation_amount), 2) AS average_compensation,
    ROUND(AVG(resolution_days), 2) AS average_resolution_days
FROM complaints
GROUP BY complaint_type
ORDER BY total_compensation DESC;
"

complaint_cost_result <- dbGetQuery(conn, complaint_cost_query)
complaint_cost_result

complaint_type,total_complaints,total_compensation,average_compensation,average_resolution_days
<chr>,<int>,<dbl>,<dbl>,<dbl>
Delay,101,1696.84,18.05,7.26
MissedPickup,64,1423.40,22.59,7.64
AppIssue,53,980.72,19.61,8.60
DriverBehaviour,51,973.06,21.15,8.16
Billing,16,381.94,23.87,7.75
Damage,15,359.73,23.98,11.33
SupportExperience,20,342.50,17.13,7.45


In [55]:
# Which booking channel generates the highest revenue?

booking_channel_query <- "
SELECT
    booking_channel,
    COUNT(order_id) AS total_orders,
    ROUND(SUM(order_value), 2) AS total_revenue,
    ROUND(AVG(order_value), 2) AS average_order_value
FROM orders
GROUP BY booking_channel
ORDER BY total_revenue DESC;
"

booking_channel_result <- dbGetQuery(conn, booking_channel_query)
booking_channel_result

booking_channel,total_orders,total_revenue,average_order_value
<chr>,<int>,<dbl>,<dbl>
App,635,58172.54,91.61
Web,269,25270.79,93.94
Phone,257,22619.09,88.01
API,64,5529.19,86.39
,25,2221.54,88.86


In [56]:
# Which vehicle types have the lowest battery health?

battery_health_query <- "
SELECT
    vehicle_type,
    COUNT(vehicle_id) AS total_vehicles,
    ROUND(AVG(battery_health_pct), 2) AS average_battery_health,
    ROUND(MIN(battery_health_pct), 2) AS lowest_battery_health,
    ROUND(MAX(battery_health_pct), 2) AS highest_battery_health
FROM vehicles
GROUP BY vehicle_type
ORDER BY average_battery_health ASC;
"

battery_health_result <- dbGetQuery(conn, battery_health_query)
battery_health_result

vehicle_type,total_vehicles,average_battery_health,lowest_battery_health,highest_battery_health
<chr>,<int>,<dbl>,<dbl>,<dbl>
Diesel,19,70.56,42.0,90.4
CargoVan,30,73.38,47.6,93.6
Hybrid,28,76.84,49.7,100.0
EV,43,82.12,63.5,100.0


In [57]:
# Which hubs have the highest average delivery fuel or charging cost?

hub_cost_query <- "
SELECT
    h.hub_id,
    h.hub_name,
    h.zone,
    COUNT(d.delivery_id) AS total_deliveries,
    ROUND(AVG(d.fuel_or_charge_cost), 2) AS average_delivery_cost,
    ROUND(SUM(d.fuel_or_charge_cost), 2) AS total_delivery_cost
FROM hubs h
JOIN deliveries d
ON h.hub_id = d.hub_id
GROUP BY h.hub_id, h.hub_name, h.zone
ORDER BY average_delivery_cost DESC;
"

hub_cost_result <- dbGetQuery(conn, hub_cost_query)
hub_cost_result

hub_id,hub_name,zone,total_deliveries,average_delivery_cost,total_delivery_cost
<chr>,<chr>,<chr>,<int>,<dbl>,<dbl>
H05,Central Core,Central,115,13.69,1573.89
H06,Airport Hub,Airport,104,13.32,1385.20
H04,West Gate,West,127,13.17,1672.21
H07,Riverside Hub,Riverside,115,12.92,1486.04
H01,North Exchange,North,136,12.76,1734.79
H03,East Dock,East,119,12.74,1516.56
H02,South Link,South,106,12.57,1331.89
H08,Midtown Relay,Central,128,11.71,1498.65


In [58]:
# Which app event types have the highest failure rate?

app_failure_query <- "
SELECT
    event_type,
    COUNT(event_id) AS total_events,
    SUM(CASE WHEN success_flag = 0 THEN 1 ELSE 0 END) AS failed_events,
    ROUND(
        SUM(CASE WHEN success_flag = 0 THEN 1 ELSE 0 END) * 100.0 / COUNT(event_id),
        2
    ) AS failure_rate_percentage,
    ROUND(AVG(api_latency_ms), 2) AS average_api_latency
FROM app_events
GROUP BY event_type
ORDER BY failure_rate_percentage DESC;
"

app_failure_result <- dbGetQuery(conn, app_failure_query)
app_failure_result

event_type,total_events,failed_events,failure_rate_percentage,average_api_latency
<chr>,<int>,<int>,<dbl>,<dbl>
chat_escalated,38,19,50.00,478.13
payment_retry,69,19,27.54,472.68
track_order,138,0,0.00,460.71
search_route,99,0,0.00,456.51
eta_refresh,105,0,0.00,452.15
delivery_instruction_update,75,0,0.00,496.29
chat_opened,88,0,0.00,478.33
cancel_attempt,28,0,0.00,417.14


In [59]:
# Which drivers have high route overrides and low customer ratings?
driver_override_rating_query <- "
SELECT
    dr.driver_id,
    dr.base_zone,
    dr.years_experience,
    dr.training_score,
    dr.driver_rating,
    COUNT(d.delivery_id) AS total_deliveries,
    SUM(d.manual_route_override_count) AS total_route_overrides,
    ROUND(AVG(d.manual_route_override_count), 2) AS average_route_overrides,
    ROUND(AVG(d.customer_rating_post_delivery), 2) AS average_customer_rating,
    SUM(CASE WHEN d.delivery_status = 'Failed' THEN 1 ELSE 0 END) AS failed_deliveries
FROM drivers dr
JOIN deliveries d
ON dr.driver_id = d.driver_id
GROUP BY
    dr.driver_id,
    dr.base_zone,
    dr.years_experience,
    dr.training_score,
    dr.driver_rating
HAVING COUNT(d.delivery_id) >= 3
ORDER BY average_route_overrides DESC, average_customer_rating ASC
LIMIT 5;
"
driver_override_rating_result <- dbGetQuery(conn, driver_override_rating_query)
driver_override_rating_result

driver_id,base_zone,years_experience,training_score,driver_rating,total_deliveries,total_route_overrides,average_route_overrides,average_customer_rating,failed_deliveries
<chr>,<chr>,<int>,<dbl>,<dbl>,<int>,<int>,<dbl>,<dbl>,<int>
D127,CENTRAL,10,61.5,4.19,6,17,2.83,4.10,0
D124,north,4,70.6,3.78,4,8,2.00,3.41,0
D085,North,9,84.5,4.11,4,8,2.00,3.42,0
D130,WEST,8,71.2,3.64,8,16,2.00,3.80,1
D062,South,10,62.4,4.48,3,6,2.00,3.82,1
